In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from PortfolioOptimization import get_report

In [2]:
max_risk = 0.15

In [3]:
output_dir = './Output/Change in Max Risk'
os.makedirs(output_dir, exist_ok=True)
output_dir = f'{output_dir}/{max_risk:.2%}'
os.makedirs(output_dir, exist_ok=True)

In [4]:
from Portfolios.GSPCBuyAndHold import BuyAndHold
# from Portfolios.EqualWeights import EqWeightPortfolio
from Portfolios.Markowitz import MarkowitzPortfolio
from Portfolios.MeanAbsDev import MeanAbsDevPortfolio

freqs = [
    # '1d',
    # '1wk',
    '1mo',
    '3mo',
    '6mo',
    '1y'
]
portfolios = {
    'S&P BnH': BuyAndHold,
    # 'Equal': EqWeightPortfolio,
    'Markowitz': MarkowitzPortfolio,
    'MAD': MeanAbsDevPortfolio,
}

reports = {}

In [5]:
for freq in freqs:
    os.makedirs(f'{output_dir}/{freq}', exist_ok=True)
    os.makedirs(f'{output_dir}/{freq}/Metrics', exist_ok=True)
    os.makedirs(f'{output_dir}/{freq}/Weights', exist_ok=True)
    all_stats = {}
    all_weights = {}
    for name, portfolio in portfolios.items():
        weights, stats = get_report(portfolio, freq=freq, max_risk=max_risk)
        if name == 'S&P BnH' or name == 'Equal':
            pass
        else:
            all_weights[name] = weights
        weights.to_excel(f'{output_dir}/{freq}/Weights/{name}.xlsx')

        if name == 'S&P BnH' or name == 'Equal':
            pass
        else:
            all_weights[name] = weights

        all_stats[name] = stats
        stats_used = list(stats.columns)

    for stat in stats_used:
        stat_graph_df = pd.concat([
                            all_stats[name][[stat]].rename(columns={stat: name})
                            for name in portfolios.keys()
                        ], axis=1)
        if stat == 'PV':
            stat_graph_df.plot(kind='line', figsize=(18, 8))
        else:
            stat_graph_df.plot(kind='bar', figsize=(18, 8))
        plt.xticks(rotation=90)
        plt.title(f"{stat.title()} Comparison for all Portfolios (Rebalanced {freq})")
        plt.savefig(f'{output_dir}/{freq}/Metrics/{stat}.png', bbox_inches='tight')
        plt.close()

    weight_comp = {
        'period': all_stats[list(all_stats.keys())[0]].index[1:]
    }
    for name in all_weights.keys():
        weights = list(all_weights[name].values / 100)
        distance_from_eq = list(map(
            lambda x: np.sum(np.abs(x - np.ones(len(weights[0])) / len(weights[0])) ** 2),
            weights
        ))
        weight_comp[name] = np.array(distance_from_eq)
    weight_comp['Inter'] = [
        np.sum((x - y) ** 2)
        for x, y in zip(list(all_weights['MAD'].values / 100), list(all_weights['Markowitz'].values / 100))
    ]
    weight_comp = pd.DataFrame.from_dict(weight_comp).set_index('period')
    weight_comp.to_excel(f'{output_dir}/{freq}/Weights/Comparison.xlsx')
    weight_comp.plot(kind='bar', figsize=(18, 8))
    plt.savefig(f'{output_dir}/{freq}/Weights/Comparison.png', bbox_inches='tight')
    plt.close()